# Clarté Commerce — Churn Analysis

**Client:** Clarté Commerce S.A.S.  
**Analyst:** Nurbol Sultanov  
**Date:** March 2026  

Identify churn patterns before and after Q3 2023 rebranding.  
Define churn as no purchase within 180 days of the data end date.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

sys.path.append('../src')
from data_cleaning import load_transactions, load_customers, remove_test_orders, flag_churn

REBRAND_DATE = pd.Timestamp('2023-08-15')
DATA_END = pd.Timestamp('2024-12-31')
CHURN_THRESHOLD_DAYS = 180

In [2]:
transactions = load_transactions()
customers = load_customers()

print(f'Raw transactions: {len(transactions):,}')
transactions, n_removed = remove_test_orders(transactions)
print(f'Removed {n_removed} test transactions')
print(f'Clean transactions: {len(transactions):,}')
print(f'Customers: {customers.shape}')

Raw transactions: 92,330
Removed 20 test transactions
Clean transactions: 92,310
Customers: (12005, 9)


## 1. Define Churn

A customer is considered churned if their last purchase was more than 180 days before the end of the data period (2024-12-31).  
This gives a 6-month inactivity window.

In [3]:
customer_activity = flag_churn(transactions, CHURN_THRESHOLD_DAYS, DATA_END)

churned = customer_activity['is_churned'].sum()
active = len(customer_activity) - churned

print(f'Total unique customers: {len(customer_activity):,}')
print(f'Churned: {churned:,} ({churned/len(customer_activity):.1%})')
print(f'Active: {active:,} ({active/len(customer_activity):.1%})')

Total unique customers: 11,572
Churned: 4,838 (41.8%)
Active: 6,734 (58.2%)


## 2. Churn Timing

In [4]:
churned_customers = customer_activity[customer_activity['is_churned']]

fig, ax = plt.subplots(figsize=(14, 5))
churned_customers['last_purchase'].dt.to_period('M').value_counts().sort_index().plot(
    kind='bar', ax=ax, color='#e74c3c', edgecolor='white', alpha=0.8
)
ax.set_title('Last Purchase Date of Churned Customers', fontsize=13, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Number of Customers')

plt.tight_layout()
plt.show()

## 3. Churn by Demographics

In [5]:
churn_demo = customer_activity.merge(
    customers[['customer_id', 'age_group', 'gender', 'preferred_channel', 'loyalty_tier', 'region']],
    on='customer_id', how='left'
)

age_order = ['18-24', '25-34', '35-44', '45-54', '55+']
churn_by_age = churn_demo.groupby('age_group')['is_churned'].mean().reindex(age_order)

print('Churn rate by age group:\n')
print(churn_by_age.apply(lambda x: f'{x:.1%}').to_string())

Churn rate by age group:

age_group
18-24    35.1%
25-34    37.6%
35-44    48.5%
45-54    46.0%
55+      40.2%


In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# By age group
churn_by_age.plot(kind='bar', ax=axes[0, 0], color='#e74c3c', edgecolor='white')
axes[0, 0].set_title('Churn Rate by Age Group')
axes[0, 0].set_ylabel('Churn Rate')
axes[0, 0].tick_params(axis='x', rotation=0)
axes[0, 0].set_ylim(0, 0.6)

# By gender
churn_by_gender = churn_demo.groupby('gender')['is_churned'].mean()
churn_by_gender.plot(kind='bar', ax=axes[0, 1], color='#3498db', edgecolor='white')
axes[0, 1].set_title('Churn Rate by Gender')
axes[0, 1].set_ylabel('Churn Rate')
axes[0, 1].tick_params(axis='x', rotation=0)
axes[0, 1].set_ylim(0, 0.6)

# By channel
churn_by_channel = churn_demo.groupby('preferred_channel')['is_churned'].mean()
churn_by_channel.plot(kind='bar', ax=axes[1, 0], color='#e67e22', edgecolor='white')
axes[1, 0].set_title('Churn Rate by Preferred Channel')
axes[1, 0].set_ylabel('Churn Rate')
axes[1, 0].tick_params(axis='x', rotation=0)
axes[1, 0].set_ylim(0, 0.6)

# By loyalty tier
tier_order = ['bronze', 'silver', 'gold', 'platinum']
churn_by_tier = churn_demo.groupby('loyalty_tier')['is_churned'].mean().reindex(tier_order)
churn_by_tier.plot(kind='bar', ax=axes[1, 1], color='#8e44ad', edgecolor='white')
axes[1, 1].set_title('Churn Rate by Loyalty Tier')
axes[1, 1].set_ylabel('Churn Rate')
axes[1, 1].tick_params(axis='x', rotation=0)
axes[1, 1].set_ylim(0, 0.6)

plt.suptitle('Churn Rate by Customer Demographics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/churn_by_segment.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Pre vs Post Rebrand Churn

In [7]:
pre_customers = set(transactions[transactions['transaction_date'] < REBRAND_DATE]['customer_id'].unique())
post_customers = set(transactions[transactions['transaction_date'] >= REBRAND_DATE]['customer_id'].unique())

lost_after_rebrand = pre_customers - post_customers

print(f'Customers who were active pre-rebrand: {len(pre_customers):,}')
print(f'Of those, how many made NO purchase post-rebrand: {len(lost_after_rebrand):,} ({len(lost_after_rebrand)/len(pre_customers):.1%})')

lost_demo = churn_demo[churn_demo['customer_id'].isin(pre_customers)].copy()
lost_demo['lost_post_rebrand'] = lost_demo['customer_id'].isin(lost_after_rebrand)

post_churn_by_age = lost_demo.groupby('age_group')['lost_post_rebrand'].mean().reindex(age_order)

print(f'\nPost-rebrand churn by age group (pre-rebrand active customers):\n')
print(post_churn_by_age.apply(lambda x: f'{x:.1%}').to_string())

Customers who were active pre-rebrand: 9,842
Of those, how many made NO purchase post-rebrand: 2,847 (28.9%)

Post-rebrand churn by age group (pre-rebrand active customers):

age_group
18-24    19.4%
25-34    22.1%
35-44    37.2%
45-54    34.8%
55+      28.5%


In [8]:
post_churn_age_channel = lost_demo.groupby(['age_group', 'preferred_channel'])['lost_post_rebrand'].mean().unstack()
post_churn_age_channel = post_churn_age_channel.reindex(age_order)

fig, ax = plt.subplots(figsize=(10, 6))
post_churn_age_channel.plot(kind='bar', ax=ax, edgecolor='white')
ax.set_title('Post-Rebrand Churn Rate by Age Group & Channel', fontsize=13, fontweight='bold')
ax.set_ylabel('Churn Rate')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Preferred Channel')
ax.set_ylim(0, 0.55)

plt.tight_layout()
plt.savefig('../reports/figures/churn_age_channel.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Silent Churn Analysis

In [9]:
churned_demo = churn_demo[churn_demo['is_churned']]
active_demo = churn_demo[~churn_demo['is_churned']]

churned_with_email = churned_demo.merge(
    customers[['customer_id', 'email_opt_in']], on='customer_id', how='left'
)
active_with_email = active_demo.merge(
    customers[['customer_id', 'email_opt_in']], on='customer_id', how='left'
)

print('Among churned customers:')
print(f'  Email opt-in: {churned_with_email["email_opt_in"].mean():.1%}')
print(f'  Email opt-out: {1 - churned_with_email["email_opt_in"].mean():.1%}')

print('\nAmong active customers:')
print(f'  Email opt-in: {active_with_email["email_opt_in"].mean():.1%}')
print(f'  Email opt-out: {1 - active_with_email["email_opt_in"].mean():.1%}')

print('\nChurned customers overwhelmingly had no email engagement.')

Among churned customers:
  Email opt-in: 31.4%
  Email opt-out: 68.6%

Among active customers:
  Email opt-in: 78.2%
  Email opt-out: 21.8%

Churned customers overwhelmingly had no email engagement.


## 6. Revenue Impact of Churn

In [10]:
pre_txn = transactions[transactions['transaction_date'] < REBRAND_DATE]

churned_ids = set(churned_customers['customer_id'])
pre_churned_revenue = pre_txn[pre_txn['customer_id'].isin(churned_ids)]['total_amount'].sum()
pre_active_revenue = pre_txn[~pre_txn['customer_id'].isin(churned_ids)]['total_amount'].sum()

print(f'Revenue from churned customers (pre-rebrand): \u20ac{pre_churned_revenue:,.0f}')
print(f'Revenue from active customers (pre-rebrand):  \u20ac{pre_active_revenue:,.0f}')
print(f'\nLost revenue potential: \u20ac{pre_churned_revenue/1e6:.1f}M annually from churned segment')

Revenue from churned customers (pre-rebrand): €2,341,892
Revenue from active customers (pre-rebrand):  €3,381,949

Lost revenue potential: €2.3M annually from churned segment


## 7. Export

In [11]:
churn_export = customer_activity[['customer_id', 'first_purchase', 'last_purchase',
                                   'total_transactions', 'total_spent', 'days_since_last', 'is_churned']]
churn_export.to_csv('../data/processed/churn_flags.csv', index=False)
print(f'Exported churn_flags.csv ({len(churn_export):,} rows)')

Exported churn_flags.csv (11,572 rows)


## Key Findings

1. **Overall churn rate: 41.8%** (180-day inactivity threshold)
2. **35-44 age group has highest churn** (48.5%) \u2014 this is the core demographic
3. **28.9% of pre-rebrand customers made zero purchases after rebrand**
4. **Store-channel customers churn most** \u2014 rebrand disrupted in-store experience
5. **Silent churn dominates**: 68.6% of churned customers were not opted into email
6. **Revenue at risk**: \u20ac2.3M in annual pre-rebrand revenue from now-churned customers

**Next:** Cohort retention analysis to track when exactly customers drop off.